# Live Momentum Trading System

Development and testing workspace. **Production code lives in the `.py` files** — this notebook is for exploring and debugging.

| File | Purpose |
|------|---------|
| `momentum_trading_system_complete.py` | Signal generation (momentum scores, portfolio selection) |
| `broker.py` | IBKR order execution via ib_insync |
| `live_trader.py` | Full rebalance logic — S&P 500 auto-trade + TSX manual instructions |
| `rebalance_job.py` | Cron entry point with logging and email notification |
| `dashboard.py` | Streamlit portfolio dashboard |
| `tsx_backtest.ipynb` | 50/50 S&P 500 + TSX backtest |

**To run a rebalance manually:**
```bash
python live_trader.py          # paper trading
python live_trader.py --live   # real money
```

**Quarterly cron fires automatically** at 9:35 AM on March/June/September/December 15.


In [20]:
""" LIVE MOMENTUM TRADING SYSTEM
Author: Xiangru Mo
Strategy: 6-month momentum, quarterly rebalance
Universe:  S&P 500 (auto-traded via IBKR) + TSX (manual)

Production files:
  broker.py                        IBKR order execution
  live_trader.py                   quarterly rebalance logic
  rebalance_job.py                 cron entry point + email notifications
  momentum_trading_system_complete.py  signal generation

This notebook is a development/testing workspace.
Run the system via: python rebalance_job.py
"""

import pandas as pd
import numpy as np
import yfinance as yf
import sqlite3
from pathlib import Path
from datetime import datetime

print("=" * 80)
print(" LIVE MOMENTUM TRADING SYSTEM")
print(" Development & Testing Workspace")
print("=" * 80)


LIVE MOMENTUM TRADING SYSTEM
Initialized: 2026-03-11 11:38:09



In [21]:
"""
SYSTEM CONFIGURATION
All parameters for live trading
"""


class LiveConfig:
    # Strategy Parameters (same as backtest)
    lookback_months = 6
    holding_months = 3
    skip_days = 21
    n_positions = 50

    # Derived
    lookback_days = lookback_months * 21
    holding_days = holding_months * 21

    # Trading Schedule
    rebalance_frequency = "quarterly"  # When to rebalance
    rebalance_months = [3, 6, 9, 12]  # End of Mar, Jun, Sep, Dec
    check_time = "15:45"  # Check daily at 3:45pm ET (before close)

    # Portfolio
    initial_capital = 100000  # $100K
    position_size_method = "equal_weight"  # Equal weight each position

    # Risk Management
    max_position_size = 0.05  # Max 5% per position (safety)
    min_position_size = 0.01  # Min 1% per position
    cash_buffer = 0.02  # Keep 2% cash for slippage

    # Data Storage
    data_dir = Path("live_trading_data")
    db_file = data_dir / "trading_system.db"

    # Broker (placeholder - will be set when connecting)
    broker = None  # 'alpaca', 'ibkr', or 'paper'

    def __repr__(self):
        return f"""
Live Trading Configuration:
--------------------------
Strategy: S&P 500 Momentum
Lookback: {self.lookback_months} months
Holding: {self.holding_months} months
Positions: {self.n_positions} stocks
Rebalance: {self.rebalance_frequency} ({self.rebalance_months})
Check Time: {self.check_time} ET

Portfolio:
Initial Capital: ${self.initial_capital:,.0f}
Position Sizing: {self.position_size_method}
Cash Buffer: {self.cash_buffer*100:.0f}%

Data Directory: {self.data_dir}
        """


# Initialize
live_config = LiveConfig()

# Create data directory
live_config.data_dir.mkdir(exist_ok=True)

print(live_config)


Live Trading Configuration:
--------------------------
Strategy: S&P 500 Momentum
Lookback: 6 months
Holding: 3 months
Positions: 50 stocks
Rebalance: quarterly ([3, 6, 9, 12])
Check Time: 15:45 ET

Portfolio:
Initial Capital: $100,000
Position Sizing: equal_weight
Cash Buffer: 2%

Data Directory: live_trading_data
        


In [22]:
"""
DATABASE MODULE
Store historical data, signals, positions, trades
"""


def init_database(db_file):
    """Initialize SQLite database with tables"""
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    # Table 1: Price Data
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS prices (
            date TEXT,
            ticker TEXT,
            close REAL,
            adj_close REAL,
            PRIMARY KEY (date, ticker)
        )
    """
    )

    # Table 2: Signals (momentum scores)
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS signals (
            date TEXT,
            ticker TEXT,
            momentum_score REAL,
            rank INTEGER,
            selected BOOLEAN,
            PRIMARY KEY (date, ticker)
        )
    """
    )

    # Table 3: Current Positions
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS positions (
            ticker TEXT PRIMARY KEY,
            shares INTEGER,
            entry_price REAL,
            entry_date TEXT,
            current_value REAL,
            last_updated TEXT
        )
    """
    )

    # Table 4: Trade History
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS trades (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            ticker TEXT,
            action TEXT,
            shares INTEGER,
            price REAL,
            value REAL,
            status TEXT
        )
    """
    )

    # Table 5: Portfolio History
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS portfolio_history (
            date TEXT PRIMARY KEY,
            total_value REAL,
            cash REAL,
            positions_value REAL,
            num_positions INTEGER
        )
    """
    )

    # Table 6: Rebalance Log
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS rebalance_log (
            date TEXT PRIMARY KEY,
            num_buys INTEGER,
            num_sells INTEGER,
            portfolio_value_before REAL,
            portfolio_value_after REAL,
            notes TEXT
        )
    """
    )

    conn.commit()
    conn.close()

    print(f"✓ Database initialized: {db_file}")
    print(
        "  Tables: prices, signals, positions, trades, portfolio_history, rebalance_log"
    )


# Initialize database
init_database(live_config.db_file)

✓ Database initialized: live_trading_data/trading_system.db
  Tables: prices, signals, positions, trades, portfolio_history, rebalance_log


In [23]:
"""
DATA PIPELINE MODULE
Fetch and update price data
"""


def get_sp500_tickers():
    """Get current S&P 500 tickers"""
    import requests
    from io import StringIO

    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    tables = pd.read_html(StringIO(response.text))
    sp500_table = tables[0]
    tickers = sp500_table["Symbol"].tolist()
    tickers = [ticker.replace(".", "-") for ticker in tickers]
    return tickers


def fetch_latest_prices(tickers, days_back=30):
    """
    Fetch recent price data

    Parameters:
    - tickers: list of ticker symbols
    - days_back: how many days of history to fetch

    Returns:
    - DataFrame with adjusted close prices
    """
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days_back)

    print(f"Fetching prices for {len(tickers)} tickers...")
    print(f"Date range: {start_date.date()} to {end_date.date()}")

    data = yf.download(
        tickers, start=start_date, end=end_date, auto_adjust=False, progress=False
    )

    adj_close = data["Adj Close"]

    # Clean up
    if isinstance(adj_close, pd.Series):
        adj_close = adj_close.to_frame()

    print(f"✓ Fetched {adj_close.shape[0]} days, {adj_close.shape[1]} stocks")
    return adj_close


def update_price_database(db_file, adj_close):
    """Save prices to database"""
    conn = sqlite3.connect(db_file)

    records = []
    for date in adj_close.index:
        for ticker in adj_close.columns:
            price = adj_close.loc[date, ticker]
            if pd.notna(price):
                records.append(
                    {
                        "date": date.strftime("%Y-%m-%d"),
                        "ticker": ticker,
                        "close": price,
                        "adj_close": price,
                    }
                )

    df = pd.DataFrame(records)
    df.to_sql("prices", conn, if_exists="append", index=False)

    conn.close()
    print(f"✓ Saved {len(records)} price records to database")


def load_price_history(db_file, days_back=252):
    """Load historical prices from database"""
    conn = sqlite3.connect(db_file)

    cutoff_date = (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d")

    query = f"""
        SELECT date, ticker, adj_close
        FROM prices
        WHERE date >= '{cutoff_date}'
        ORDER BY date
    """

    df = pd.read_sql(query, conn)
    conn.close()

    # Pivot to wide format
    pivot = df.pivot(index="date", columns="ticker", values="adj_close")
    pivot.index = pd.to_datetime(pivot.index)

    print(f"✓ Loaded {pivot.shape[0]} days, {pivot.shape[1]} stocks from database")
    return pivot


print("=" * 80)
print("DATA PIPELINE MODULE")
print("=" * 80)
print("✓ Data pipeline loaded")

DATA PIPELINE MODULE
✓ Data pipeline loaded


In [24]:
"""
SIGNAL GENERATION MODULE
Calculate momentum scores and select portfolio
"""


def calculate_live_momentum_scores(adj_close, lookback_days=126, skip_days=21):
    """
    Calculate momentum scores using latest data

    Returns:
    - DataFrame with tickers, scores, and ranks
    """
    # Use most recent date
    latest_date = adj_close.index[-1]

    # Need enough history
    if len(adj_close) < lookback_days:
        print(f"⚠ Warning: Only {len(adj_close)} days available, need {lookback_days}")
        return pd.DataFrame()

    # Calculate momentum
    prices_lookback = adj_close.iloc[-lookback_days]
    prices_skip = adj_close.iloc[-skip_days]

    momentum_scores = (prices_skip / prices_lookback) - 1
    momentum_scores = momentum_scores.dropna()

    # Create DataFrame with rankings
    df = pd.DataFrame(
        {"ticker": momentum_scores.index, "momentum_score": momentum_scores.values}
    )

    df = df.sort_values("momentum_score", ascending=False).reset_index(drop=True)
    df["rank"] = df.index + 1
    df["date"] = latest_date.strftime("%Y-%m-%d")

    print(f"\n✓ Calculated momentum for {len(df)} stocks")
    print(f"  Date: {latest_date.date()}")
    print(f"  Top 5: {df.head(5)['ticker'].tolist()}")
    print(f"  Top score: {df.iloc[0]['momentum_score']*100:.2f}%")

    return df


def select_portfolio(momentum_df, n_positions=50):
    """
    Select top N stocks for portfolio

    Returns:
    - List of selected tickers
    """
    selected = momentum_df.head(n_positions).copy()
    selected["selected"] = True

    print(f"✓ Selected {len(selected)} stocks for portfolio")
    return selected["ticker"].tolist(), selected


def save_signals_to_db(db_file, signals_df):
    """Save signals to database"""
    conn = sqlite3.connect(db_file)
    # Filter out signals that already exist in the database
    existing_dates = pd.read_sql("SELECT DISTINCT date FROM signals", conn)[
        "date"
    ].tolist()
    new_signals = signals_df[~signals_df["date"].isin(existing_dates)]
    if not new_signals.empty:
        new_signals.to_sql("signals", conn, if_exists="append", index=False)
        print(f"✓ Saved {len(new_signals)} new signals to database")
    else:
        print("✓ Signals already exist for this date, skipping save")
    conn.close()


print("=" * 80)
print("SIGNAL GENERATION MODULE")
print("=" * 80)
print("✓ Signal generator loaded")

SIGNAL GENERATION MODULE
✓ Signal generator loaded


In [25]:
"""
ORDER GENERATION MODULE
Generate buy/sell orders for rebalancing
"""


def get_current_positions(db_file):
    """Load current positions from database"""
    conn = sqlite3.connect(db_file)

    query = "SELECT ticker, shares, entry_price, entry_date FROM positions"
    df = pd.read_sql(query, conn)
    conn.close()

    if df.empty:
        print("No current positions")
        return {}

    positions = df.set_index("ticker").to_dict("index")
    print(f"✓ Loaded {len(positions)} current positions")
    return positions


def generate_rebalance_orders(
    target_tickers, current_positions, portfolio_value, n_positions=50, cash_buffer=0.02
):
    """
    Generate orders to rebalance portfolio

    Parameters:
    - target_tickers: list of tickers to hold
    - current_positions: dict of current holdings
    - portfolio_value: total portfolio value
    - n_positions: target number of positions
    - cash_buffer: % to keep as cash

    Returns:
    - buy_orders: list of {ticker, shares, value}
    - sell_orders: list of {ticker, shares, value}
    """
    # Calculate target position size
    investable_capital = portfolio_value * (1 - cash_buffer)
    target_position_value = investable_capital / n_positions

    # Current tickers
    current_tickers = set(current_positions.keys())
    target_tickers_set = set(target_tickers)

    # Determine actions
    to_sell = current_tickers - target_tickers_set  # In portfolio but not in target
    to_buy = target_tickers_set - current_tickers  # In target but not in portfolio
    to_hold = current_tickers & target_tickers_set  # In both

    buy_orders = []
    sell_orders = []

    # Generate sell orders
    for ticker in to_sell:
        shares = current_positions[ticker]["shares"]
        sell_orders.append(
            {
                "ticker": ticker,
                "action": "SELL",
                "shares": shares,
                "reason": "Not in target portfolio",
            }
        )

    # Generate buy orders (will need current prices to calculate shares)
    for ticker in to_buy:
        buy_orders.append(
            {
                "ticker": ticker,
                "action": "BUY",
                "target_value": target_position_value,
                "reason": "New position",
            }
        )

    print(f"\n{'='*60}")
    print("REBALANCE ORDERS")
    print(f"{'='*60}")
    print(f"Portfolio Value: ${portfolio_value:,.2f}")
    print(f"Target Position Size: ${target_position_value:,.2f}")
    print(f"\nTo Sell: {len(sell_orders)} positions")
    print(f"To Buy: {len(buy_orders)} positions")
    print(f"To Hold: {len(to_hold)} positions")
    print(f"{'='*60}")

    return buy_orders, sell_orders


print("=" * 80)
print("ORDER GENERATION MODULE")
print("=" * 80)
print("✓ Order generator loaded")

ORDER GENERATION MODULE
✓ Order generator loaded


In [26]:
"""
PORTFOLIO MANAGER
Track positions, calculate portfolio value, manage state
"""


def initialize_portfolio(db_file, initial_capital):
    """Initialize portfolio with starting cash"""
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    # Check if already initialized
    cursor.execute("SELECT COUNT(*) FROM portfolio_history")
    count = cursor.fetchone()[0]

    if count == 0:
        # Record initial state
        cursor.execute(
            """
            INSERT INTO portfolio_history 
            (date, total_value, cash, positions_value, num_positions)
            VALUES (?, ?, ?, ?, ?)
        """,
            (
                datetime.now().strftime("%Y-%m-%d"),
                initial_capital,
                initial_capital,
                0,
                0,
            ),
        )
        conn.commit()
        print(f"✓ Portfolio initialized with ${initial_capital:,.2f}")
    else:
        print(f"✓ Portfolio already initialized ({count} records)")

    conn.close()


def get_portfolio_value(db_file, current_prices):
    """
    Calculate current portfolio value

    Parameters:
    - db_file: path to database
    - current_prices: dict of {ticker: price}

    Returns:
    - dict with total_value, cash, positions_value, positions
    """
    conn = sqlite3.connect(db_file)

    # Get current positions
    positions_df = pd.read_sql("SELECT * FROM positions", conn)

    # Get latest portfolio state
    portfolio_df = pd.read_sql(
        "SELECT * FROM portfolio_history ORDER BY date DESC LIMIT 1", conn
    )

    conn.close()

    if portfolio_df.empty:
        return {
            "total_value": 0,
            "cash": 0,
            "positions_value": 0,
            "num_positions": 0,
            "positions": {},
        }

    cash = portfolio_df.iloc[0]["cash"]

    # Calculate positions value
    positions_value = 0
    positions_dict = {}

    for _, pos in positions_df.iterrows():
        ticker = pos["ticker"]
        shares = pos["shares"]
        current_price = current_prices.get(ticker, pos["entry_price"])

        value = shares * current_price
        positions_value += value

        positions_dict[ticker] = {
            "shares": shares,
            "entry_price": pos["entry_price"],
            "current_price": current_price,
            "value": value,
            "pnl": value - (shares * pos["entry_price"]),
            "pnl_pct": (current_price / pos["entry_price"] - 1) * 100,
        }

    total_value = cash + positions_value

    return {
        "total_value": total_value,
        "cash": cash,
        "positions_value": positions_value,
        "num_positions": len(positions_dict),
        "positions": positions_dict,
    }


def update_portfolio_history(db_file, portfolio_state):
    """Save portfolio snapshot to database"""
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    cursor.execute(
        """
        INSERT OR REPLACE INTO portfolio_history 
        (date, total_value, cash, positions_value, num_positions)
        VALUES (?, ?, ?, ?, ?)
    """,
        (
            datetime.now().strftime("%Y-%m-%d"),
            portfolio_state["total_value"],
            portfolio_state["cash"],
            portfolio_state["positions_value"],
            portfolio_state["num_positions"],
        ),
    )

    conn.commit()
    conn.close()
    print("✓ Portfolio history updated")


print("=" * 80)
print("PORTFOLIO MANAGER")
print("=" * 80)
print("✓ Portfolio manager loaded")

# Initialize portfolio
initialize_portfolio(live_config.db_file, live_config.initial_capital)

PORTFOLIO MANAGER
✓ Portfolio manager loaded
✓ Portfolio already initialized (1 records)


In [27]:
"""
MAIN ORCHESTRATOR
Coordinates all modules for live trading
"""


class MomentumTradingSystem:
    def __init__(self, config):
        self.config = config
        self.db_file = config.db_file

    def check_if_rebalance_needed(self):
        """Check if today is a rebalance day"""
        today = datetime.now()

        # Check if end of quarter month
        if today.month not in self.config.rebalance_months:
            return False, "Not a rebalance month"

        # Check if last business day of month
        next_day = today + timedelta(days=1)
        if next_day.month == today.month:
            return False, "Not end of month yet"

        return True, "Rebalance day!"

    def run_signal_generation(self):
        """Generate momentum signals"""
        print("\n" + "=" * 80)
        print("GENERATING SIGNALS")
        print("=" * 80)

        # Fetch recent data
        tickers = get_sp500_tickers()
        adj_close = fetch_latest_prices(tickers, days_back=300)

        # Calculate momentum scores
        momentum_df = calculate_live_momentum_scores(
            adj_close, self.config.lookback_days, self.config.skip_days
        )

        # Select portfolio
        target_tickers, signals_df = select_portfolio(
            momentum_df, self.config.n_positions
        )

        # Save to database
        save_signals_to_db(self.db_file, signals_df)

        return target_tickers, adj_close

    def run_order_generation(self, target_tickers, current_prices_df):
        """Generate rebalance orders"""
        print("\n" + "=" * 80)
        print("GENERATING ORDERS")
        print("=" * 80)

        # Get current positions
        current_positions = get_current_positions(self.db_file)

        # Get current prices as dict
        current_prices = current_prices_df.iloc[-1].to_dict()

        # Calculate portfolio value
        portfolio_state = get_portfolio_value(self.db_file, current_prices)

        print("\nCurrent Portfolio State:")
        print(f"  Total Value: ${portfolio_state['total_value']:,.2f}")
        print(f"  Cash: ${portfolio_state['cash']:,.2f}")
        print(f"  Positions Value: ${portfolio_state['positions_value']:,.2f}")
        print(f"  Positions: {portfolio_state['num_positions']}")

        # Generate orders
        buy_orders, sell_orders = generate_rebalance_orders(
            target_tickers,
            current_positions,
            portfolio_state["total_value"],
            self.config.n_positions,
            self.config.cash_buffer,
        )

        # Calculate shares for buy orders
        for order in buy_orders:
            ticker = order["ticker"]
            current_price = current_prices.get(ticker, 0)
            if current_price > 0:
                shares = int(order["target_value"] / current_price)
                order["shares"] = shares
                order["estimated_cost"] = shares * current_price

        return buy_orders, sell_orders, portfolio_state

    def display_orders(self, buy_orders, sell_orders):
        """Display orders for review"""
        print("\n" + "=" * 80)
        print("ORDER SUMMARY")
        print("=" * 80)

        if sell_orders:
            print(f"\nSELL ORDERS ({len(sell_orders)}):")
            print(f"{'Ticker':<10} {'Shares':<10} {'Reason':<30}")
            print("-" * 50)
            for order in sell_orders[:10]:  # Show first 10
                print(
                    f"{order['ticker']:<10} {order['shares']:<10} {order['reason']:<30}"
                )
            if len(sell_orders) > 10:
                print(f"... and {len(sell_orders) - 10} more")

        if buy_orders:
            print(f"\nBUY ORDERS ({len(buy_orders)}):")
            print(f"{'Ticker':<10} {'Shares':<10} {'Est. Cost':<15} {'Reason':<30}")
            print("-" * 65)
            for order in buy_orders[:10]:  # Show first 10
                cost = order.get("estimated_cost", 0)
                print(
                    f"{order['ticker']:<10} {order['shares']:<10} ${cost:>12,.2f}  {order['reason']:<30}"
                )
            if len(buy_orders) > 10:
                print(f"... and {len(buy_orders) - 10} more")

        print("=" * 80)

    def run_daily_check(self):
        """Main daily routine"""
        print("\n" + "=" * 80)
        print(f"DAILY CHECK - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 80)

        # Check if rebalance needed
        should_rebalance, reason = self.check_if_rebalance_needed()
        print(f"Rebalance needed: {should_rebalance} - {reason}")

        if should_rebalance:
            # Generate signals
            target_tickers, current_prices = self.run_signal_generation()

            # Generate orders
            buy_orders, sell_orders, portfolio_state = self.run_order_generation(
                target_tickers, current_prices
            )

            # Display for review
            self.display_orders(buy_orders, sell_orders)

            print("\n⚠ Orders generated but NOT executed (no broker connected)")
            print("Connect broker to execute trades automatically")

            return {
                "rebalance": True,
                "buy_orders": buy_orders,
                "sell_orders": sell_orders,
                "portfolio_state": portfolio_state,
            }
        else:
            print("No action needed today")
            return {"rebalance": False}


print("=" * 80)
print("MAIN ORCHESTRATOR")
print("=" * 80)

# Initialize system
trading_system = MomentumTradingSystem(live_config)
print("✓ Trading system initialized")

MAIN ORCHESTRATOR
✓ Trading system initialized


In [28]:
"""
TEST RUN
Simulate a daily check
"""

print("=" * 80)
print("RUNNING TEST SIMULATION")
print("=" * 80)

# Run daily check
result = trading_system.run_daily_check()

print("\n" + "=" * 80)
print("TEST COMPLETE")
print("=" * 80)

RUNNING TEST SIMULATION

DAILY CHECK - 2026-03-11 11:38:18
Rebalance needed: False - Not end of month yet
No action needed today

TEST COMPLETE


In [30]:
"""
SCHEDULER
Run system automatically every day at market close
"""

import schedule  # noqa: E402
import threading  # noqa: E402


def scheduled_daily_check():
    """Function to run daily"""
    print(f"\n{'='*80}")
    print(f"SCHEDULED CHECK: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}")

    try:
        result = trading_system.run_daily_check()

        if result.get("rebalance"):
            print("\n🔔 REBALANCE EXECUTED!")
            print(f"Buy orders: {len(result['buy_orders'])}")
            print(f"Sell orders: {len(result['sell_orders'])}")

            # TODO: Send notification (email, Telegram, etc.)
        else:
            print("✓ No rebalance needed, standing pat")

    except Exception as e:
        print(f"❌ Error during scheduled check: {e}")


def run_scheduler():
    """Run scheduler in background"""
    # Schedule daily check at 3:45 PM ET (before market close)
    schedule.every().day.at(live_config.check_time).do(scheduled_daily_check)

    print("✓ Scheduler started")
    print(f"  Daily check at: {live_config.check_time} ET")
    print(f"  Rebalance months: {live_config.rebalance_months}")

    while True:
        schedule.run_pending()
        time.sleep(60)  # Check every minute


def start_scheduler_background():
    """Start scheduler in background thread"""
    scheduler_thread = threading.Thread(target=run_scheduler, daemon=True)
    scheduler_thread.start()
    print("✓ Scheduler running in background")
    return scheduler_thread


print("=" * 80)
print("SCHEDULER MODULE")
print("=" * 80)
print("✓ Scheduler loaded")
print("\nTo start: scheduler_thread = start_scheduler_background()")
print("To test now: scheduled_daily_check()")

SCHEDULER MODULE
✓ Scheduler loaded

To start: scheduler_thread = start_scheduler_background()
To test now: scheduled_daily_check()


In [48]:
"""
PORTFOLIO DASHBOARD
View current positions and performance
"""


def display_portfolio_dashboard(db_file):
    """Display comprehensive portfolio view"""
    conn = sqlite3.connect(db_file)

    # Get positions
    positions = pd.read_sql("SELECT * FROM positions", conn)

    # Get portfolio history
    history = pd.read_sql(
        "SELECT * FROM portfolio_history ORDER BY date DESC LIMIT 5", conn
    )

    # Get recent trades

    # Get rebalance log
    rebalances = pd.read_sql(
        "SELECT * FROM rebalance_log ORDER BY date DESC LIMIT 5", conn
    )

    conn.close()

    print("\n" + "=" * 80)
    print("PORTFOLIO DASHBOARD")
    print("=" * 80)

    # Current positions
    if not positions.empty:
        print(f"\n📊 CURRENT POSITIONS ({len(positions)} stocks)")
        print("-" * 80)
        print(
            f"{'Ticker':<8} {'Shares':<8} {'Entry $':<12} {'Current $':<12} {'Entry Date':<12}"
        )
        print("-" * 80)
        for _, pos in positions.head(10).iterrows():
            print(
                f"{pos['ticker']:<8} {pos['shares']:<8} ${pos['entry_price']:>10.2f} "
                f"${pos['current_value']/pos['shares']:>10.2f} {pos['entry_date']:<12}"
            )
        if len(positions) > 10:
            print(f"... and {len(positions) - 10} more positions")
    else:
        print("\n📊 No positions currently")

    # Portfolio value
    if not history.empty:
        latest = history.iloc[0]
        print("\n💰 PORTFOLIO VALUE")
        print("-" * 80)
        print(f"Total Value:      ${latest['total_value']:>15,.2f}")
        print(f"Cash:             ${latest['cash']:>15,.2f}")
        print(f"Positions Value:  ${latest['positions_value']:>15,.2f}")
        print(f"Number of Stocks: {latest['num_positions']:>15}")

    # Rebalance history
    if not rebalances.empty:
        print("\n🔄 REBALANCE HISTORY")
        print("-" * 80)
        print(f"{'Date':<12} {'Buys':<8} {'Sells':<8} {'Notes':<40}")
        print("-" * 80)
        for _, reb in rebalances.iterrows():
            print(
                f"{reb['date']:<12} {reb['num_buys']:<8} {reb['num_sells']:<8} {reb['notes']:<40}"
            )

    print("=" * 80)


# Display dashboard
display_portfolio_dashboard(live_config.db_file)


PORTFOLIO DASHBOARD

📊 CURRENT POSITIONS (50 stocks)
--------------------------------------------------------------------------------
Ticker   Shares   Entry $      Current $    Entry Date  
--------------------------------------------------------------------------------


ZeroDivisionError: float division by zero

In [49]:
"""
SAVE SYSTEM CONFIGURATION
Export configuration and summary
"""


def save_system_summary(config, filename="system_summary.json"):
    """Save system configuration and status"""

    summary = {
        "system_name": "S&P 500 Momentum Trading System",
        "created": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "strategy": {
            "lookback_months": config.lookback_months,
            "holding_months": config.holding_months,
            "skip_days": config.skip_days,
            "n_positions": config.n_positions,
            "rebalance_frequency": config.rebalance_frequency,
        },
        "risk_management": {
            "max_position_size": config.max_position_size,
            "cash_buffer": config.cash_buffer,
        },
        "portfolio": {
            "initial_capital": config.initial_capital,
            "position_sizing": config.position_size_method,
        },
        "execution": {
            "broker": config.broker.__class__.__name__ if config.broker else "None",
            "check_time": config.check_time,
            "rebalance_months": config.rebalance_months,
        },
        "status": "Active - Paper Trading",
    }

    # Save to file
    filepath = config.data_dir / filename
    with open(filepath, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"✓ System summary saved to: {filepath}")

    # Also print summary
    print("\n" + "=" * 80)
    print("SYSTEM SUMMARY")
    print("=" * 80)
    print(json.dumps(summary, indent=2))
    print("=" * 80)

    return summary


# Save summary
system_summary = save_system_summary(live_config)

✓ System summary saved to: live_trading_data/system_summary.json

SYSTEM SUMMARY
{
  "system_name": "S&P 500 Momentum Trading System",
  "created": "2026-03-11 13:31:03",
  "strategy": {
    "lookback_months": 6,
    "holding_months": 3,
    "skip_days": 21,
    "n_positions": 50,
    "rebalance_frequency": "quarterly"
  },
  "risk_management": {
    "max_position_size": 0.05,
    "cash_buffer": 0.02
  },
  "portfolio": {
    "initial_capital": 100000,
    "position_sizing": "equal_weight"
  },
  "execution": {
    "broker": "ImprovedPaperBroker",
    "check_time": "15:45",
    "rebalance_months": [
      3,
      6,
      9,
      12
    ]
  },
  "status": "Active - Paper Trading"
}


In [ ]:
"""
SYSTEM HEALTH CHECK
Verify everything is working
"""


def system_health_check(config):
    """Run comprehensive system check"""

    print("=" * 80)
    print("SYSTEM HEALTH CHECK")
    print("=" * 80)

    checks = []

    # Check 1: Database exists
    if config.db_file.exists():
        checks.append(("✅", "Database", "Connected"))
    else:
        checks.append(("❌", "Database", "Not found"))

    # Check 2: Database tables
    conn = sqlite3.connect(config.db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cursor.fetchall()]
    conn.close()

    expected_tables = [
        "prices",
        "signals",
        "positions",
        "trades",
        "portfolio_history",
        "rebalance_log",
    ]
    if all(t in tables for t in expected_tables):
        checks.append(("✅", "Database Tables", f"{len(tables)} tables"))
    else:
        checks.append(("⚠️", "Database Tables", "Missing tables"))

    # Check 3: Broker connection
    if config.broker and config.broker.connected:
        checks.append(("✅", "Broker", config.broker.__class__.__name__))
    else:
        checks.append(("❌", "Broker", "Not connected"))

    # Check 4: Current positions
    conn = sqlite3.connect(config.db_file)
    positions = pd.read_sql("SELECT COUNT(*) as count FROM positions", conn)
    conn.close()

    pos_count = positions.iloc[0]["count"]
    if pos_count > 0:
        checks.append(("✅", "Positions", f"{pos_count} stocks"))
    else:
        checks.append(("⚠️", "Positions", "No positions"))

    # Check 5: Strategy config
    checks.append(
        ("✅", "Strategy", f"J={config.lookback_months}, K={config.holding_months}")
    )

    # Check 6: Rebalance schedule
    checks.append(("✅", "Schedule", f"Quarterly at {config.check_time} ET"))

    # Display results
    print(f"\n{'Status':<8} {'Component':<20} {'Details':<30}")
    print("-" * 80)
    for status, component, details in checks:
        print(f"{status:<8} {component:<20} {details:<30}")

    print("\n" + "=" * 80)

    # Overall status
    failed = sum(1 for c in checks if c[0] == "❌")
    warnings = sum(1 for c in checks if c[0] == "⚠️")

    if failed == 0 and warnings == 0:
        print("✅ System Status: ALL SYSTEMS GO")
    elif failed == 0:
        print(f"⚠️  System Status: OPERATIONAL ({warnings} warnings)")
    else:
        print(f"❌ System Status: ISSUES DETECTED ({failed} failures)")

    print("=" * 80)


# Run health check
system_health_check(live_config)

SYSTEM HEALTH CHECK

Status   Component            Details                       
--------------------------------------------------------------------------------
✅        Database             Connected                     
✅        Database Tables      7 tables                      
✅        Broker               ImprovedPaperBroker           
✅        Positions            50 stocks                     
✅        Strategy             J=6, K=3                      
✅        Schedule             Quarterly at 15:45 ET         

✅ System Status: ALL SYSTEMS GO


In [42]:
"""
CREATE PROJECT STRUCTURE
Organize code for Git
"""

from pathlib import Path  # noqa: E402

# Define project structure
project_structure = {
    "README.md": None,
    ".gitignore": None,
    "requirements.txt": None,
    "config/": {
        "__init__.py": None,
        "config.py": None,
    },
    "src/": {
        "__init__.py": None,
        "data_pipeline.py": None,
        "signal_generator.py": None,
        "portfolio_manager.py": None,
        "order_executor.py": None,
        "brokers.py": None,
    },
    "backtests/": {
        "momentum_backtest.ipynb": None,
    },
    "live_trading/": {
        "main.py": None,
        "scheduler.py": None,
    },
    "tests/": {
        "__init__.py": None,
    },
    "data/": {
        ".gitkeep": None,
    },
    "notebooks/": {
        "research.ipynb": None,
    },
}


def create_project_structure(base_path="momentum_trading_system"):
    """Create clean project structure"""
    base = Path(base_path)
    base.mkdir(exist_ok=True)

    def create_dirs(structure, parent):
        for name, content in structure.items():
            path = parent / name
            if isinstance(content, dict):
                path.mkdir(exist_ok=True)
                create_dirs(content, path)
            else:
                path.touch()

    create_dirs(project_structure, base)
    print(f"✓ Created project structure at: {base}")
    return base


# Create structure
project_dir = create_project_structure()

print("\nProject Structure:")
print("momentum_trading_system/")
print("├── README.md")
print("├── requirements.txt")
print("├── .gitignore")
print("├── config/")
print("│   └── config.py")
print("├── src/")
print("│   ├── data_pipeline.py")
print("│   ├── signal_generator.py")
print("│   ├── portfolio_manager.py")
print("│   ├── order_executor.py")
print("│   └── brokers.py")
print("├── backtests/")
print("│   └── momentum_backtest.ipynb")
print("├── live_trading/")
print("│   ├── main.py")
print("│   └── scheduler.py")
print("├── data/")
print("├── notebooks/")
print("└── tests/")

✓ Created project structure at: momentum_trading_system

Project Structure:
momentum_trading_system/
├── README.md
├── requirements.txt
├── .gitignore
├── config/
│   └── config.py
├── src/
│   ├── data_pipeline.py
│   ├── signal_generator.py
│   ├── portfolio_manager.py
│   ├── order_executor.py
│   └── brokers.py
├── backtests/
│   └── momentum_backtest.ipynb
├── live_trading/
│   ├── main.py
│   └── scheduler.py
├── data/
├── notebooks/
└── tests/


In [42]:
"""
GENERATE README
"""

readme_content = """# S&P 500 Momentum Trading System

Automated quantitative momentum trading system for S&P 500 stocks.

## 📊 Strategy Overview

- **Universe:** S&P 500 stocks
- **Signal:** 6-month momentum (J=6)
- **Holding Period:** 3 months (quarterly rebalancing)
- **Skip Period:** 1 month (avoid reversal)
- **Portfolio:** Top 50 momentum stocks, equal-weighted

## 📈 Performance (Backtested 2016-2026)

- **Annualized Return:** 27.31%
- **Sharpe Ratio:** 1.23
- **Max Drawdown:** -20.30%
- **Win Rate:** 76.9%

## 🏗️ Architecture
```
momentum_trading_system/
├── src/              # Core trading modules
├── config/           # Configuration
├── backtests/        # Backtest notebooks
├── live_trading/     # Live trading scripts
├── data/             # Local data storage
└── tests/            # Unit tests
```

## 🚀 Quick Start

### Installation
```bash
pip install -r requirements.txt
```

### Backtest
```python
# Run backtest notebook
jupyter notebook backtests/momentum_backtest.ipynb
```

### Live Trading (Paper)
```python
# Start paper trading
python live_trading/main.py --mode paper
```

## 📦 Dependencies

- pandas
- numpy
- yfinance
- matplotlib
- seaborn
- schedule
- alpaca-trade-api (optional)
- ib_insync (optional)

## ⚙️ Configuration

Edit `config/config.py` to customize:
- Strategy parameters (J, K, skip period)
- Portfolio size
- Rebalancing frequency
- Broker settings

## 🔐 Security

**Never commit:**
- API keys
- Secret keys
- Database files
- Trade history

Use environment variables for sensitive data.

## 📊 Rebalancing Schedule

- **Frequency:** Quarterly (end of March, June, September, December)
- **Check Time:** 3:45 PM ET daily
- **Execution:** Market close

## 🧪 Testing
```bash
# Run tests
python -m pytest tests/
```

## 📝 Research

See `notebooks/research.ipynb` for:
- Parameter optimization
- Factor analysis
- Performance attribution

## ⚠️ Disclaimer

This is educational software. Trading involves risk. Past performance does not guarantee future results. Use at your own risk.

## 📧 Contact

Author: Mo Xiang  
University of Waterloo, FARM '26

## 📄 License

MIT License

---

**Built with:** Python 3.13 | pandas | yfinance
"""

with open(project_dir / "README.md", "w") as f:
    f.write(readme_content)

print("✓ README.md created")

✓ README.md created


In [43]:
"""
GENERATE .gitignore
"""

gitignore_content = """# Python
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
env/
venv/
.venv/
ENV/
*.egg-info/
dist/
build/

# Jupyter
.ipynb_checkpoints/
*.ipynb_checkpoints

# Data
*.csv
*.pkl
*.db
*.sqlite
*.sqlite3
data/
live_trading_data/
*.json

# API Keys & Secrets
.env
*.key
*_secret.txt
config_local.py

# Logs
*.log
logs/

# OS
.DS_Store
Thumbs.db

# IDE
.vscode/
.idea/
*.swp
*.swo

# Backtest outputs
backtest_results/
performance_charts/
*.png

# Temporary files
temp/
tmp/
*.tmp

# Keep structure
!data/.gitkeep
!tests/.gitkeep
"""

with open(project_dir / ".gitignore", "w") as f:
    f.write(gitignore_content)

print("✓ .gitignore created")

✓ .gitignore created


In [44]:
"""
GENERATE requirements.txt
"""

requirements = """# Core dependencies
pandas>=2.0.0
numpy>=1.24.0
yfinance>=0.2.0

# Visualization
matplotlib>=3.7.0
seaborn>=0.12.0

# Scheduling
schedule>=1.2.0

# Database
sqlite3

# Optional: Brokers
ib_insync>=0.9.86

# Optional: Monitoring
streamlit>=1.28.0

# Development
jupyter>=1.0.0
pytest>=7.4.0
"""

with open(project_dir / "requirements.txt", "w") as f:
    f.write(requirements)

print("✓ requirements.txt created")

✓ requirements.txt created


In [40]:
"""
COPY YOUR WORK
Move notebooks to project directory
"""


# Files to copy (adjust paths to match your actual files)
files_to_copy = {
    # Backtest notebook - you'll need to specify actual path
    # 'momentum_trading_system.ipynb': project_dir / 'backtests' / 'momentum_backtest.ipynb',
    # Live trading notebook
    # 'live_trading_system.ipynb': project_dir / 'notebooks' / 'live_trading_development.ipynb',
}

print("Manual step: Copy your notebooks to the project directory")
print("\nCopy:")
print(
    f"  momentum_trading_system.ipynb → {project_dir}/backtests/momentum_backtest.ipynb"
)
print(
    f"  live_trading_system.ipynb → {project_dir}/notebooks/live_trading_development.ipynb"
)

Manual step: Copy your notebooks to the project directory

Copy:
  momentum_trading_system.ipynb → momentum_trading_system/backtests/momentum_backtest.ipynb
  live_trading_system.ipynb → momentum_trading_system/notebooks/live_trading_development.ipynb


In [54]:
"""
UPDATE DATABASE WITH FRESH PRICES
Fetch latest prices and update positions
"""

import sqlite3  # noqa: E402

print("Fetching current market prices...")

# Get all tickers from positions
conn = sqlite3.connect(live_config.db_file)
positions = pd.read_sql("SELECT ticker, shares FROM positions", conn)
conn.close()

if positions.empty:
    print("No positions to update")
else:
    tickers = positions["ticker"].tolist()

    # Fetch latest prices (real-time)
    print(f"Fetching prices for {len(tickers)} tickers...")

    current_prices = {}
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            # Get the most recent price (could be live or last close)
            info = stock.info
            price = (
                info.get("currentPrice")
                or info.get("regularMarketPrice")
                or info.get("previousClose")
            )

            if price:
                current_prices[ticker] = price
                print(f"  {ticker}: ${price:.2f}")
        except Exception as e:
            print(f"  ⚠️  {ticker}: Could not fetch price - {e}")

    # Update positions table with current prices
    conn = sqlite3.connect(live_config.db_file)
    cursor = conn.cursor()

    for ticker, price in current_prices.items():
        shares = positions[positions["ticker"] == ticker]["shares"].iloc[0]
        current_value = shares * price

        cursor.execute(
            """
            UPDATE positions 
            SET current_value = ?, last_updated = ?
            WHERE ticker = ?
        """,
            (current_value, datetime.now().strftime("%Y-%m-%d %H:%M:%S"), ticker),
        )

    conn.commit()
    conn.close()

    print(f"\n✓ Updated {len(current_prices)} positions with current prices")
    print("\nRefresh your dashboard to see updated prices!")

Fetching current market prices...
Fetching prices for 50 tickers...
  SNDK: $643.64
  WDC: $270.73
  MU: $422.10
  TER: $304.01
  CIEN: $334.39
  ALB: $166.82
  STX: $385.19
  WBD: $27.85
  LRCX: $220.04
  INTC: $48.04
  AMAT: $350.64
  GLW: $133.28
  FIX: $1405.47
  CAT: $709.62
  LUV: $41.58
  MRNA: $55.77
  FDX: $360.47
  HAL: $35.70
  JBHT: $212.44
  CHRW: $177.91
  NEM: $116.44
  KLAC: $1479.13
  STLD: $183.50
  CMI: $557.13
  HII: $417.08
  VTRS: $14.02
  CAH: $215.64
  TPR: $146.73
  DD: $45.73
  FCX: $61.08
  DG: $144.72
  SLB: $47.99
  UPS: $100.58
  BG: $121.42
  MRK: $116.55
  ALGN: $176.38
  MPWR: $1066.16
  LMT: $650.35
  REGN: $777.51
  KEYS: $283.37
  LLY: $998.18
  GM: $74.75
  AMD: $204.69
  NUE: $170.56
  XOM: $151.79
  PWR: $569.40
  MCK: $938.36
  EXPD: $141.26
  TRGP: $234.70
  JNJ: $242.66

✓ Updated 50 positions with current prices

Refresh your dashboard to see updated prices!
